In [ ]:
import random
from html import escape
import ipywidgets as widgets
from IPython.display import display

try:
    from questions import QUESTIONS
except ModuleNotFoundError as error:
    if error.name != "questions":
        raise
    from google.colab import files

    print("Envie o arquivo questions.py do projeto para carregar as perguntas.")
    uploaded = files.upload()
    if "questions.py" not in uploaded:
        raise FileNotFoundError("O arquivo questions.py não foi enviado.")
    import importlib

    importlib.invalidate_caches()
    from questions import QUESTIONS

PRIZES = [1_000, 2_500, 5_000, 10_000, 25_000, 50_000, 100_000, 250_000, 500_000, 1_000_000]
if len(QUESTIONS) != len(PRIZES):
    raise ValueError("O banco de perguntas precisa ter exatamente 10 níveis.")

print(f"Banco carregado: {sum(len(level) for level in QUESTIONS)} perguntas em {len(QUESTIONS)} níveis.")

In [ ]:
STARTING_SKIPS = 3
state = {}

title = widgets.HTML("<h2>Jogo do Milhão</h2>")
progress = widgets.IntProgress(value=1, min=1, max=len(QUESTIONS), description="Nível:")
prize_label = widgets.Label()
earned_label = widgets.Label()
question_label = widgets.HTML()
message_label = widgets.HTML()
hint_label = widgets.Label()
answer_buttons = [
    widgets.Button(description="", layout=widgets.Layout(width="100%"))
    for _ in range(4)
]
stop_button = widgets.Button(description="Parar e ficar com o prêmio")
skip_button = widgets.Button(description="Pular pergunta")
half_button = widgets.Button(description="Eliminar duas respostas")
hint_button = widgets.Button(description="Ver dica")
restart_button = widgets.Button(description="Jogar novamente", button_style="success")


def money(value):
    return f"R$ {value:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


def select_question(exclude_id=None):
    available = [
        question for question in QUESTIONS[state["level"]]
        if question["id"] != exclude_id
    ]
    if not available:
        state["question"] = None
        state["message"] = "Este nível ainda não tem perguntas disponíveis."
        state["finished"] = True
        return False
    state["question"] = random.choice(available)
    state["order"] = list(range(len(state["question"]["options"])))
    random.shuffle(state["order"])
    state["hidden"] = []
    state["hint_visible"] = False
    return True


def render():
    question = state["question"]
    progress.value = min(state["level"] + 1, len(QUESTIONS))
    prize_label.value = "Prêmio do nível: " + money(PRIZES[state["level"]])
    earned_label.value = "Prêmio garantido: " + money(state["earned"])
    skip_button.description = f"Pular pergunta ({state['skips']} restantes)"
    skip_button.disabled = state["finished"] or state["skips"] == 0
    half_button.disabled = state["finished"] or state["half_used"]
    hint_button.disabled = (
        state["finished"] or state["hint_used"] or not question or not question.get("hint")
    )
    stop_button.disabled = state["finished"]
    restart_button.layout.display = "" if state["finished"] else "none"

    if state["finished"]:
        question_label.value = ""
        message_label.value = f"<h3>{state['message']}</h3>"
        hint_label.value = ""
        for button in answer_buttons:
            button.disabled = True
            button.description = ""
        return

    question_label.value = f"<h3>{escape(question['text'])}</h3>"
    message_label.value = ""
    hint_label.value = (
        "Dica: " + question["hint"] if state["hint_visible"] else ""
    )
    for displayed_index, button in enumerate(answer_buttons):
        option_index = state["order"][displayed_index]
        button.disabled = displayed_index in state["hidden"]
        answer_text = question["options"][option_index]
        if displayed_index in state["hidden"]:
            answer_text = "(eliminada)"
        button.description = f"{'ABCD'[displayed_index]}. {answer_text}"


def finish(message, prize):
    state["finished"] = True
    state["earned"] = prize
    state["message"] = f"{message}<br>Prêmio final: {money(prize)}"
    render()


def answer(displayed_index):
    if state["finished"] or displayed_index in state["hidden"]:
        return
    question = state["question"]
    selected_option = state["order"][displayed_index]
    if selected_option != question["correct_index"]:
        finish("Resposta errada! Você perdeu tudo.", 0)
        return

    state["earned"] = PRIZES[state["level"]]
    if state["level"] == len(QUESTIONS) - 1:
        finish("Parabéns! Você chegou ao milhão!", state["earned"])
        return
    state["level"] += 1
    select_question()
    render()


def stop_game(_button):
    if not state["finished"]:
        finish("Você decidiu parar.", state["earned"])


def skip_question(_button):
    if not state["finished"] and state["skips"] > 0:
        if select_question(exclude_id=state["question"]["id"]):
            state["skips"] -= 1
        render()


def eliminate_answers(_button):
    if not state["finished"] and not state["half_used"]:
        correct_display_index = state["order"].index(state["question"]["correct_index"])
        wrong_indexes = [index for index in range(4) if index != correct_display_index]
        state["hidden"] = random.sample(wrong_indexes, 2)
        state["half_used"] = True
        render()


def show_hint(_button):
    if not state["finished"] and not state["hint_used"] and state["question"].get("hint"):
        state["hint_used"] = True
        state["hint_visible"] = True
        render()


def start_game(_button=None):
    state.update(
        level=0,
        earned=0,
        skips=STARTING_SKIPS,
        half_used=False,
        hint_used=False,
        finished=False,
        message="",
    )
    select_question()
    render()


for index, button in enumerate(answer_buttons):
    button.on_click(lambda _button, answer_index=index: answer(answer_index))
stop_button.on_click(stop_game)
skip_button.on_click(skip_question)
half_button.on_click(eliminate_answers)
hint_button.on_click(show_hint)
restart_button.on_click(start_game)

display(widgets.VBox([
    title,
    progress,
    prize_label,
    earned_label,
    question_label,
    *answer_buttons,
    widgets.HBox([stop_button, skip_button]),
    widgets.HBox([half_button, hint_button]),
    hint_label,
    message_label,
    restart_button,
]))
start_game()